# Spanish Transcritption

In [15]:
!pip install pyannote.audio==3.1.1 --upgrade

In [2]:
!pip install faster_whisper

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [3]:
!pip install pyannote.audio


In [4]:
!pip install noisereduce

## Imports


In [17]:
import numpy as np
import torch
import noisereduce as nr
import soundfile as sf
from pyannote.audio import Pipeline
from faster_whisper import WhisperModel
from pydub import AudioSegment
import tempfile, os

## Load and Resample

In [5]:
def load_and_resample(path: str, target_sr: int = 16000):
    audio = AudioSegment.from_file(path)
    audio = audio.set_channels(1).set_frame_rate(target_sr)
    samples = np.array(audio.get_array_of_samples()).astype(np.float32)
    samples /= np.iinfo(audio.array_type).max   # normalize to [-1, 1]
    return samples, target_sr

## VAD

In [27]:
def run_vad(samples: np.ndarray, sr: int):
    model, utils = torch.hub.load(
        repo_or_dir="snakers4/silero-vad",
        model="silero_vad",
        force_reload=False
    )
    (get_speech_timestamps, _, read_audio, *_) = utils

    audio_tensor = torch.FloatTensor(samples)
    speech_timestamps = get_speech_timestamps(
        audio_tensor, model,
        sampling_rate=sr,
        threshold=0.5,
        min_speech_duration_ms=250,
        min_silence_duration_ms=500,
    )
    segments = [(t["start"] / sr, t["end"] / sr) for t in speech_timestamps]
    print(f"[VAD] Found {len(segments)} speech segments")
    return segments

## Concatenate only the VAD speech chunks

In [7]:
def extract_speech_audio(samples: np.ndarray, sr: int, segments):
    chunks = [samples[int(s * sr): int(e * sr)] for s, e in segments]
    speech_audio = np.concatenate(chunks) if chunks else samples
    print(f"[VAD] Kept {len(speech_audio)/sr:.1f}s of speech "
          f"from {len(samples)/sr:.1f}s total")
    return speech_audio

## Noise Reduction

In [8]:
def reduce_noise(samples: np.ndarray, sr: int):
    """Spectral noise reduction (noisereduce)."""
    denoised = nr.reduce_noise(y=samples, sr=sr, stationary=False, prop_decrease=0.8)
    print("[Noise Reduction] Done")
    return denoised

## Transcribe

In [26]:
def transcribe(samples: np.ndarray, sr: int, speech_segments: list):
    model = WhisperModel("medium", device="cuda", compute_type="float16")

    results = []
    for i, (start, end) in enumerate(speech_segments):
        chunk = samples[int(start * sr): int(end * sr)]

        # Skip very short chunks
        if len(chunk) / sr < 0.5:
            continue

        segments, info = model.transcribe(
            chunk,
            language=None,        # auto-detect per chunk
            beam_size=5,
            vad_filter=False,
        )

        for seg in segments:
            results.append({
                "start": start + seg.start,   # offset by chunk start
                "end":   start + seg.end,
                "text":  seg.text.strip(),
                "lang":  info.language,
            })
            print(f"  [{start + seg.start:.1f}s → start + {seg.end:.1f}s] "
                  f"[{info.language}] {seg.text.strip()}")

    return results

## Keeping only Spanish

In [10]:
def filter_spanish(results):
    """Keep only segments detected as Spanish (useful if language=None)."""
    spanish = [r for r in results if r["lang"] == "es"]
    print(f"[Filter] {len(spanish)}/{len(results)} segments are Spanish")
    return spanish

## Pipeline

In [28]:
# ─── CONFIG ───────────────────────────────────────────────────────────────────
AUDIO_PATH     = "/content/drive/MyDrive/spanish/038010_EIT-2A.mp3"   # any format
TARGET_SR      = 16000              # Whisper expects 16kHz
HF_TOKEN       = ""         # for pyannote VAD
TRANSCRIBE_LANG = None                    # None = auto-detect; "es" = force Spanish
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # 1. Load + resample
    audio, sr = load_and_resample(AUDIO_PATH, TARGET_SR)

    # 2. VAD
    speech_segments = run_vad(audio, sr)

    # 3. Noise reduce full audio (before chunking)
    audio = reduce_noise(audio, sr)

    # 4. Transcribe each chunk individually with language detection
    results = transcribe(audio, sr, speech_segments)

    # 5. Filter Spanish
    spanish_results = [r for r in results if r["lang"] == "es"]
    print(f"\n[Filter] {len(spanish_results)}/{len(results)} segments are Spanish")

    full_transcript = " ".join(r["text"] for r in spanish_results)
    print("\n─── SPANISH TRANSCRIPT ───")
    print(full_transcript)

Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


[VAD] Found 46 speech segments
[Noise Reduction] Done
  [74.1s → start + 1.4s] [en] we drove to the park.
  [81.6s → start + 2.0s] [en] I'll call her tomorrow night.
  [89.2s → start + 2.0s] [en] You can buy meat at the butcher's shop.
  [97.5s → start + 2.4s] [en] My brother just bought a brand new computer.
  [107.7s → start + 2.6s] [en] Sometimes they take their dog for a walk in the park.
  [118.9s → start + 3.1s] [en] We're going to play volleyball at the gym that I told you about.
  [165.3s → start + 2.0s] [hi] के रुख खोड़ान में अल्पेलू
  [172.6s → start + 1.6s] [es] El libro está en la mesa.
  [180.0s → start + 2.0s] [es] El caro lo tiene pelo.
  [188.1s → start + 1.6s] [en] I'll see you tomorrow.
  [197.2s → start + 2.4s] [en] ¿Qué dices ustedes que van a hacer hoy?
  [206.3s → start + 2.4s] [en] I hope you know how to drive very well.
  [216.1s → start + 3.0s] [la] las kayes de esta ciudad son muy anchas.
  [227.5s → start + 2.6s] [en] Maybe tomorrow will be the day.
  [238.7s